In [ ]:
%run ./imports.py

In [ ]:
minimal_deviation_path = "data/minimal_deviation.csv"
prefix_filtered_output = "data/campus_trace_prefixes_filtered_on_minrtt.pkl"
prefixes_rtts_geo_path = "data/campus_traces_prefixes_rtts_geo.pkl"
countries_convex_hull_path = "data/countries_convex_hull_path.pkl"
minrttdev_enriched_path = "data/minrttdev_enriched.pkl"
prefixes_rtts_geo_usa_path = "data/campus_traces_prefixes_rtts_geo_usa.pkl"
json_hull_mainland_shifted_path = "data/country_hull_mainland_shifted_coordinates.json"
json_prefixes_path = "data/prefix_geoloc_coordinates.json"
prefix_rtt_profiling_path = "data/prefix_rtt_profiling_path.pkl"
prefix_rtt_profiling_fixed_windows_path = "data/prefix_rtt_profiling_fixed_windows_path.pkl"
prefix_rtt_profiling_variable_windows_path = "data/prefix_rtt_profiling_variable_windows_path.pkl"
prefix_optimal_attacks_path = "data/prefix_optimal_attacks.pkl"

In [ ]:
df_prefixes_with_rtts_geo = pd.read_pickle(prefixes_rtts_geo_path)
print(df_prefixes_with_rtts_geo.shape[0])

In [ ]:
df_prefixes_with_rtts_geo.head(n=1)

### Prefix and byte breakdown

In [ ]:
def conn_country_stats(df, conn, country):
    num_total = df.shape[0]
    byt_total = df["Prefix_Total_Bytes"].sum()
    rtt_total = df["RTT_Count"].sum()
    print(f"No. of prefixes: {num_total}, bytes: {mu.rnd(byt_total/10**9, 1)} GB, RTT count: {mu.rnd(rtt_total/10**6, 1)} M")
    df_conn = df[df['Connection_Type_Unique'].apply(lambda lst: len(lst)==1 and conn in lst)]
    num_conn = df_conn.shape[0]
    byt_conn = df_conn["Prefix_Total_Bytes"].sum()
    rtt_conn = df_conn["RTT_Count"].sum()
    print(f"No. of {conn} prefixes: {num_conn} ({mu.rnd(num_conn*100.0/num_total, 2)}%), "
          + f"bytes: {mu.rnd(byt_conn/10**9, 2)} GB ({mu.rnd(byt_conn*100.0/byt_total, 2)}%), "
          + f"RTT count: {mu.rnd(rtt_conn/10**6, 1)} M ({mu.rnd(rtt_conn*100.0/rtt_total, 2)}%)")
    df_conn_country = df_conn[df_conn["Country"] == country]
    num_conn_country = df_conn_country.shape[0]
    byt_conn_country = df_conn_country["Prefix_Total_Bytes"].sum()
    rtt_conn_country = df_conn_country["RTT_Count"].sum()
    print(f"No. of {conn} prefixes in {country}: {num_conn_country} ({mu.rnd(num_conn_country*100.0/num_conn, 2)}%), "
          + f"bytes: {mu.rnd(byt_conn_country/10**9, 2)} GB ({mu.rnd(byt_conn_country*100.0/byt_conn, 2)}%), "
          + f"RTT count: {mu.rnd(rtt_conn_country/10**6, 1)} M ({mu.rnd(rtt_conn_country*100.0/rtt_conn, 2)}%)")

In [ ]:
conn_country_stats(df_prefixes_with_rtts_geo, "CISO", "United States of America")
print()
conn_country_stats(df_prefixes_with_rtts_geo, "SICO", "United States of America")

### Filter in prefixes with at least N seconds of total duration (end time - start time) and M samples

In [ ]:
MIN_WINDOW_COUNT = 10
MIN_WINDOW_SIZE  = 0.25
MAX_WINDOW_SIZE  = 60
MIN_SAMPLE_COUNT_PER_WINDOW = 5

In [ ]:
df_prefixes_duration_filtered = df_prefixes_with_rtts_geo[
    (df_prefixes_with_rtts_geo["Prefix_Duration_s"] >= MIN_WINDOW_COUNT * MIN_WINDOW_SIZE)
    & (df_prefixes_with_rtts_geo["RTT_Count"] >= MIN_WINDOW_COUNT * MIN_SAMPLE_COUNT_PER_WINDOW)
].copy()
df_prefixes_duration_filtered.head(n=1)
print("Filtering based on prefix duration")
conn_country_stats(df_prefixes_duration_filtered, "CISO", "United States of America")
print()
conn_country_stats(df_prefixes_duration_filtered, "SICO", "United States of America")

### Divide into profiling and detection phases based on the no. of RTT samples

In [ ]:
def split_into_phases(row):
    ts    = row['ACK_Timestamp']
    flows = row['Flow_ID']
    rtts  = row['RTT_ms']
    n = len(ts)
    if n < MIN_WINDOW_COUNT * MIN_SAMPLE_COUNT_PER_WINDOW:
        return ([], [], [], [], [], [])
    mid = (n + 1) // 2
    # First half: Profiling, Second half: Detection
    return (
        ts[:mid],     # ACK_Timestamp_Profiling
        ts[mid:],     # ACK_Timestamp_Detection
        flows[:mid],  # Flow_ID_Profiling
        flows[mid:],  # Flow_ID_Detection
        rtts[:mid],   # RTT_ms_Profiling
        rtts[mid:]    # RTT_ms_Detection
    )

In [ ]:
df_prefixes_phases = df_prefixes_duration_filtered.copy()
df_prefixes_phases[
    [
        'ACK_Timestamp_Profiling', 'ACK_Timestamp_Detection',
        'Flow_ID_Profiling',       'Flow_ID_Detection',
        'RTT_ms_Profiling',        'RTT_ms_Detection',
    ]
] = df_prefixes_phases.apply(split_into_phases, axis=1, result_type='expand')
df_prefixes_phases["Profiling_Min_RTT_ms"] = df_prefixes_phases["RTT_ms_Profiling"].apply(min)
df_prefixes_phases.drop(columns=['ACK_Timestamp', 'Flow_ID', 'RTT_ms'], inplace=True)
df_prefixes_phases.head(n=1)

### Collect profiling phase stats

#### Fixed window size per prefix
Note: The right fixed window size can only be computed if we have access to the profiling phase samples in software.

In [ ]:
def compute_fixed_window_size_per_prefix_stats(row):
    # extract and sort timestamps (and align Flow_ID and RTT_ms)
    ts   = np.asarray(row['ACK_Timestamp_Profiling'])
    # fids = np.asarray(row['Flow_ID_Profiling'])
    rtts = np.asarray(row['RTT_ms_Profiling'])

    # early exit if not enough total samples
    if len(ts) < (MIN_WINDOW_COUNT // 2) * MIN_SAMPLE_COUNT_PER_WINDOW:
        return -1, -1

    ts_min, ts_max = ts[0], ts[-1]

    # try each window size
    for W in np.arange(MIN_WINDOW_SIZE,
                       MAX_WINDOW_SIZE + 0.01,
                       MIN_WINDOW_SIZE):

        # align the first edge to the lower quarter‐second grid
        start_edge = np.floor(ts_min / MIN_WINDOW_SIZE) * MIN_WINDOW_SIZE
        # build bins until beyond ts_max
        edges = np.arange(start_edge,
                          ts_max + (2*W),
                          W)
        # if the number of edges is too small, give up on this row
        if len(edges) < MIN_WINDOW_COUNT // 2:
            return -1, -1

        # count samples per bin
        counts, _ = np.histogram(ts, bins=edges)
        if np.sum(counts >= MIN_SAMPLE_COUNT_PER_WINDOW) < MIN_WINDOW_COUNT // 2:
            continue

        # we found the minimal W that works: now collect stats
        # assign each timestamp to a bin index
        bin_idx = np.digitize(ts, edges) - 1  # 0…len(edges)-2
        # unique_flows = []
        min_rtts     = []
        for i, c in enumerate(counts):
            idxs = np.where(bin_idx == i)[0]
            if c >= MIN_SAMPLE_COUNT_PER_WINDOW:
                # unique_flows.append(int(np.unique(fids[idxs]).size))
                min_rtts.append(float(np.min(rtts[idxs])))

        # Compute max(min_RTT_in_window)
        max_min_rtt = max(min_rtts)

        return float(W), max_min_rtt

    # no window size satisfied the constraints
    return -1, -1

In [ ]:
df_prefixes_profiling_fixedwinsize = df_prefixes_phases.copy()
df_prefixes_profiling_fixedwinsize[[
    'Profiling_Fixed_Window_Size_s',
    'Profiling_Fixed_Max_Min_RTT_per_Window_ms'
]] = (
    df_prefixes_profiling_fixedwinsize.parallel_apply(compute_fixed_window_size_per_prefix_stats, axis=1, result_type='expand')
)
df_prefixes_profiling_fixedwinsize.head(n=1)

#### Variable window size per prefix
Note: This method is hardware-amenable.

In [ ]:
def compute_variable_window_size_per_prefix_stats(row):
    # extract and sort timestamps (and align Flow_ID and RTT_ms)
    ts   = np.asarray(row['ACK_Timestamp_Profiling'])
    # fids = np.asarray(row['Flow_ID_Profiling'])
    rtts = np.asarray(row['RTT_ms_Profiling'])

    # early exit if not enough total samples
    n = len(ts)
    if n < (MIN_WINDOW_COUNT // 2) * MIN_SAMPLE_COUNT_PER_WINDOW:
        return [], -1

    # 2) initialize the first window edge to the global 0.25 s grid at or before ts_min
    ts_min = ts[0]
    current_start = math.floor(ts_min / MIN_WINDOW_SIZE) * MIN_WINDOW_SIZE

    Window_Sizes_s = []
    Min_RTT_per_Window = []

    # 3) slide through the series
    while True:
        idx_start = np.searchsorted(ts, current_start, side='left')
        if idx_start >= n:
            break

        # find minimal W such that [current_start, current_start+W) has ≥ MIN_SAMPLE_COUNT_PER_WINDOW samples
        found = False
        for W in np.arange(MIN_WINDOW_SIZE,
                           MAX_WINDOW_SIZE + 0.01,
                           MIN_WINDOW_SIZE):
            end = current_start + W
            idx_end = np.searchsorted(ts, end, side='right')
            if (idx_end - idx_start) >= MIN_SAMPLE_COUNT_PER_WINDOW:
                found = True
                break

        if not found:
            # give up if no W ≤ MAX_WINDOW_SIZE works
            # add empty to indicate break
            Window_Sizes_s.append(-1)
        else:
            # record this window's size and stats
            Window_Sizes_s.append(float(W))
            # window_ts = ts[idx_start:idx_end]
            # window_fids = fids[idx_start:idx_end]
            window_rtts = rtts[idx_start:idx_end]
            Min_RTT_per_Window.append(float(np.min(window_rtts)))

        # move start to the next timestamp beyond this window, aligned up on the 0.25 s grid
        if idx_end >= n:
            break
        next_ts = ts[idx_end]
        current_start = math.floor(next_ts / MIN_WINDOW_SIZE) * MIN_WINDOW_SIZE

    if len(Min_RTT_per_Window) >= MIN_WINDOW_COUNT // 2:
        # Compute max(min_RTT_in_window)
        max_min_rtt = max(Min_RTT_per_Window)
        return Window_Sizes_s, max_min_rtt

    return [], -1

In [ ]:
df_prefixes_profiling = df_prefixes_profiling_fixedwinsize.copy()
df_prefixes_profiling[[
    'Profiling_Variable_Window_Sizes_s',
    'Profiling_Variable_Max_Min_RTT_per_Window_ms']] = (
    df_prefixes_profiling.parallel_apply(compute_variable_window_size_per_prefix_stats, axis=1, result_type='expand')
)
df_prefixes_profiling.drop(columns=['ACK_Timestamp_Profiling', 'Flow_ID_Profiling', 'RTT_ms_Profiling'], inplace=True)
df_prefixes_profiling.head(n=1)

In [ ]:
df_prefixes_profiling.to_pickle(prefix_rtt_profiling_path)

### Filter based on insufficient data

In [ ]:
print("Filtering based on fixed window sizes")
conn_country_stats(df_prefixes_profiling[df_prefixes_profiling["Profiling_Fixed_Window_Size_s"] > 0], "CISO", "United States of America")
print()
conn_country_stats(df_prefixes_profiling[df_prefixes_profiling["Profiling_Fixed_Window_Size_s"] > 0], "SICO", "United States of America")

In [ ]:
print("Filtering based on variable window sizes")
conn_country_stats(df_prefixes_profiling[df_prefixes_profiling["Profiling_Variable_Max_Min_RTT_per_Window_ms"] >= 0], "CISO", "United States of America")
print()
conn_country_stats(df_prefixes_profiling[df_prefixes_profiling["Profiling_Variable_Max_Min_RTT_per_Window_ms"] >= 0], "SICO", "United States of America")

In [ ]:
df_prefixes_profiling_fixedwindows    = df_prefixes_profiling[df_prefixes_profiling["Profiling_Fixed_Window_Size_s"] > 0].copy()
df_prefixes_profiling_variablewindows = df_prefixes_profiling[df_prefixes_profiling["Profiling_Variable_Max_Min_RTT_per_Window_ms"] >= 0].copy()

In [ ]:
df_prefixes_profiling_fixedwindows.to_pickle(prefix_rtt_profiling_fixed_windows_path)
df_prefixes_profiling_variablewindows.to_pickle(prefix_rtt_profiling_variable_windows_path)

#### Compare window sizes

In [ ]:
fixed_sizes = df_prefixes_profiling_fixedwindows["Profiling_Fixed_Window_Size_s"].tolist()
variable_sizes = [window
                  for windows in df_prefixes_profiling_variablewindows["Profiling_Variable_Window_Sizes_s"].tolist()
                  for window in windows if window > 0]

pu.cdfs([fixed_sizes, variable_sizes],
        {"xlabel": "Window Size (s)", "ylabel": "CDF", "curvelabels": ["Fixed window size", "Variable window size"],
         "loc": "lower right"})

### Prefix to country, distance, and bytes mappings

In [ ]:
prefix_country_map = df_prefixes_profiling.set_index('Destination_Prefix')["Country"].to_dict()
prefix_distance_map = df_prefixes_profiling.set_index('Destination_Prefix')["Geodesic_Distance_km"].to_dict()
prefix_bytes_map = df_prefixes_profiling.set_index('Destination_Prefix')["Prefix_Total_Bytes"].to_dict()
prefix_lat_map = df_prefixes_profiling.set_index('Destination_Prefix')["Latitude"].to_dict()
prefix_lon_map = df_prefixes_profiling.set_index('Destination_Prefix')["Longitude"].to_dict()
prefix_profiling_minrtt_map = df_prefixes_profiling.set_index('Destination_Prefix')["Profiling_Min_RTT_ms"].to_dict()

### Minimum deviation for US-based prefixes to all other countries

In [ ]:
C_OPTICAL_FIBER_KM_PER_MS = (2/3) * (299792458 / 10**6)
print(f"Speed of light in optical fiber: {round(C_OPTICAL_FIBER_KM_PER_MS, 2)} km/ms")

In [ ]:
PRINCETON_LATLON = (40.343899, -74.660049)

In [ ]:
def save_prefix_coords_as_json(df, json_path):
    coords = {}
    prefixes = sorted(df["Destination_Prefix"].tolist())
    for prefix in prefixes:
        coords[prefix] = []
        row_prefix = df[df["Destination_Prefix"] == prefix]
        coords[prefix].append(row_prefix["Country"].iloc[0])
        coords[prefix].append(list(PRINCETON_LATLON))
        coords[prefix].append([row_prefix["Latitude"].iloc[0], row_prefix["Longitude"].iloc[0]])
    
    with open(json_path, "w") as fp:
        json.dump(coords, fp)

In [ ]:
save_prefix_coords_as_json(df_prefixes_profiling[
                               df_prefixes_profiling["Country"] == "United States of America"], json_prefixes_path)

In [ ]:
%%bash -s "$json_hull_mainland_shifted_path" "$json_prefixes_path" "$prefix_optimal_attacks_path"
./cpp/geolocation_based_analysis/BUILD/min_post_distance_prefixes "$1" "$2" "$3"

In [ ]:
df_prefix_optimal_attacks = pd.read_csv(prefix_optimal_attacks_path)
df_prefix_optimal_attacks["PreAttack_ms"] = df_prefix_optimal_attacks["Prefix"].apply(
    lambda p: prefix_profiling_minrtt_map[p])
df_prefix_optimal_attacks["PostAttack_ms"] = df_prefix_optimal_attacks["PostAttack_km"].apply(lambda d: d/C_OPTICAL_FIBER_KM_PER_MS)
df_prefix_optimal_attacks["MinDev_ms"] = df_prefix_optimal_attacks["PostAttack_ms"] - df_prefix_optimal_attacks["PreAttack_ms"]
df_prefix_optimal_attacks.head(n=1)

In [ ]:
df_prefix_optimal_attacks.to_pickle(prefix_optimal_attacks_path)

In [ ]:
prefix_attacker_post_map = {}
prefix_attacker_dev_map = {}
for prefix, attacker, post, mindev in df_prefix_optimal_attacks[[
        "Prefix", "Attacker", "PostAttack_ms", "MinDev_ms"]].itertuples(index=False, name=None):
    if prefix not in prefix_attacker_post_map:
        prefix_attacker_post_map[prefix] = {}
    if prefix not in prefix_attacker_dev_map:
        prefix_attacker_dev_map[prefix] = {}
    if attacker not in prefix_attacker_post_map[prefix]:
        prefix_attacker_post_map[prefix][attacker] = post
    if attacker not in prefix_attacker_dev_map[prefix]:
        prefix_attacker_dev_map[prefix][attacker] = max(0, mindev)

## Proceed with variable window sizes from this point on

### Break US-based prefixes down into CISO and SICO prefixes

In [ ]:
sns_colors = list(sns.color_palette("bright"))

#### CISO US-based prefixes

In [ ]:
us_ciso_prefixes = df_prefixes_profiling_variablewindows[(df_prefixes_profiling_variablewindows[
                    "Connection_Type_Unique"].apply(lambda lst: len(lst)==1 and "CISO" in lst))
                    & (df_prefixes_profiling_variablewindows["Country"] == "United States of America")]["Destination_Prefix"].tolist()
us_ciso_prefixes = sorted(us_ciso_prefixes)
print(f"No. of US-based CISO prefixes: {len(us_ciso_prefixes)}")

#### SICO US-based prefixes

In [ ]:
us_sico_prefixes = df_prefixes_profiling_variablewindows[(df_prefixes_profiling_variablewindows[
                    "Connection_Type_Unique"].apply(lambda lst: len(lst)==1 and "SICO" in lst))
                    & (df_prefixes_profiling_variablewindows["Country"] == "United States of America")]["Destination_Prefix"].tolist()
us_sico_prefixes = sorted(us_sico_prefixes)
print(f"No. of US-based SICO prefixes: {len(us_sico_prefixes)}")

### Absolute threshold-based coverage

In [ ]:
def compute_theory_coverage(prefix_map, prefixes):
    passing = {}
    coverages = []
    total_coverage = 0
    total_attacks = 0
    for prefix in prefixes:
        passing[prefix] = []
        dev_prefix = [prefix_map[prefix][attacker] for attacker in prefix_map[prefix]]
        count = sum(1 for dev in dev_prefix if dev >= 1)
        total_coverage += count
        total_attacks  += len(dev_prefix)
        coverages.append(count * 100.0 / len(dev_prefix))
        for attacker in prefix_map[prefix]:
            if prefix_map[prefix][attacker] >= 1:
                passing[prefix].append(attacker)
    return coverages, passing, (total_coverage * 100.0 / total_attacks)

In [ ]:
coverages_ciso, passing_ciso, total_coverage_ciso = compute_theory_coverage(prefix_attacker_dev_map, us_ciso_prefixes)
coverages_sico, passing_sico, total_coverage_sico = compute_theory_coverage(prefix_attacker_dev_map, us_sico_prefixes)
coverages_ciso = sorted(coverages_ciso, reverse=True)
coverages_sico = sorted(coverages_sico, reverse=True)
print("Total coverage:")
print(f"\tCISO: {total_coverage_ciso}%")
print(f"\tSICO: {total_coverage_sico}%")

In [ ]:
pct_prefixes = {"ciso": [], "sico": []}
pct_attacks  = {"ciso": [], "sico": []}

for ptype in pct_prefixes:
    if ptype == "ciso":
        coverages_ptype = coverages_ciso.copy()
    else:
        coverages_ptype = coverages_sico.copy()
    for c in range(1, len(coverages_ptype)+1):
        a = coverages_ptype[c-1]
        pct_prefixes[ptype].append(c * 100.0 / len(coverages_ptype))
        pct_attacks[ptype].append(a)

In [ ]:
pu.lineplots([pct_prefixes["ciso"], pct_prefixes["sico"]], [pct_attacks["ciso"], pct_attacks["sico"]], {
        "figsize": (8, 6),
        "colors": sns_colors,
        "linestyles": ["-", "--"],
        "loc": "lower left",
        "curvelabels": ["Defend Campus Clients", "Defend Campus Servers"],
        "xlim": (-5, 105),
        "ylim": (-5, 105),
        "xlabel": "US Prefixes (%)",
        "ylabel": "Non-US Attacks (%)",
        "plot_path": "plots/prefixes_vs_attacks_theory.pdf"
})

### Surge threshold-based coverage

### Divide RTTs into Windows of Time

In [ ]:
def divide_into_windows_with_reltime(flow_list, time_list_abs, rtt_list, window_size):
    windows_flow = []
    windows_time = []
    windows_rtt = []
    current_window_flow = []
    current_window_time = []
    current_window_rtt = []

    utc_time_min = math.floor(time_list_abs[0])
    time_list = [(t - utc_time_min) for t in time_list_abs]
    
    for flow, time, rtt in zip(flow_list, time_list, rtt_list):
        if not current_window_time:
            current_window_flow.append(flow)
            current_window_time.append(time)
            current_window_rtt.append(rtt)
        else:
            window_start = current_window_time[0]
            window_number = math.floor(time / window_size)
            start_window_number = math.floor(window_start / window_size)
            
            if window_number == start_window_number:
                current_window_flow.append(flow)
                current_window_time.append(time)
                current_window_rtt.append(rtt)
            else:
                windows_flow.append(current_window_flow)
                windows_time.append(current_window_time)
                windows_rtt.append(current_window_rtt)
                current_window_flow = [flow]
                current_window_time = [time]
                current_window_rtt = [rtt]
    
    if current_window_time:
        windows_flow.append(current_window_flow)
        windows_time.append(current_window_time)
        windows_rtt.append(current_window_rtt)
    
    return windows_flow, windows_time, windows_rtt

In [ ]:
def extract_minimums_and_window_len(flows, tstamps, rtts):
    tstamp_mins = []
    rtt_mins = []
    window_lens = []
    flow_lens = []
    for f, t, r in zip(flows, tstamps, rtts):
        loc = r.index(min(r))
        tstamp_mins.append(t[loc])
        rtt_mins.append(r[loc])
        window_lens.append(len(r))
        flow_lens.append({fid: f.count(fid) for fid in set(f)})
    return tstamp_mins, rtt_mins, window_lens, flow_lens

In [ ]:
def minrtts(rtts_dict, prefix, win_size_time, win_size_samples):
    fid_series = [i[0] for i in rtts_dict[prefix]]
    ts_series  = [i[1] for i in rtts_dict[prefix]]
    tsr_series = [(i - math.floor(ts_series[0])) for i in ts_series]
    rtt_series = [i[2] for i in rtts_dict[prefix]]
    flows_1, time_1, rtt_1 = divide_into_windows_with_reltime(fid_series, ts_series, rtt_series, win_size_time)
    min_time, min_rtts, win_len, flows_len = extract_minimums_and_window_len(flows_1, time_1, rtt_1)

    x_min_time = []
    y_min_rtts = []
    for t, r, w in zip(min_time, min_rtts, win_len):
        if w >= win_size_samples:
            x_min_time.append(t)
            y_min_rtts.append(r)

    return tsr_series, x_min_time, rtt_series, y_min_rtts

In [ ]:
def filter_minimums(time, rtt):
    tstamps = []
    minrtts = []
    for t1, t2, r1, r2, w1, w2 in zip(time[:-1], time[1:], rtt[:-1], rtt[1:], winlen[:-1], winlen[1:]):
        if w1 >= window_size_samples and w2 >= window_size_samples:
            tstamps.append(t1)
            minrtts.append(r1)
    if winlen[-1] >= window_size_samples:
        tstamps.append(time[-1])
        minrtts.append(rtt[-1])
    return tstamps, minrtts

In [ ]:
def get_distance_from_princeton(country):
    src_coord = (40.343899, -74.660049)
    dst_coord = (float(row['Latitude']), float(row['Longitude']))
    distance = geodesic(src_coord, dst_coord).km
    return distance

### Window-based coverage

In [ ]:
def compute_windowmin_based_deviation(df, S_latlon, D_latlon, country, samples):

    def coord(point):
        return (point.y, point.x)
    
    sampled_geos = df[df['Country'] == country].Smallest_Geometry_sampled_shifted.iloc[0]
    shifted_hulls = []
    for geo in sampled_geos:
        shifted_hulls.append(geo.convex_hull)

    min_dist_post = np.inf
    S_opt, D_opt, A_opt = None, None, None

    S, D = Point((S_latlon[1], S_latlon[0])), Point((D_latlon[1], D_latlon[0]))
    if S == D:
        line_SD = Point(S)
    else:
        line_SD = LineString([S, D])

    A_candidates = []
    for hull in shifted_hulls:
        point1, point2 = nearest_points(line_SD, hull)
        distance = point1.distance(point2)
        A_candidates.append((distance, point1, point2))
    A = min(A_candidates, key=lambda item: item[0])[-1]
            
    d_SD = geodesic(coord(S), coord(D)).km
    d_SA = geodesic(coord(S), coord(A)).km
    d_DA = geodesic(coord(D), coord(A)).km
    dist_post = d_SA + d_DA + d_SD
            
    if dist_post < min_dist_post:
        min_dist_post = dist_post
        S_opt, D_opt, A_opt = coord(S), coord(D), coord(A)

    flows = []
    times = []
    rtts = []
    for f, t, r in samples:
        flows.append(f)
        times.append(t)
        rtts.append(t)
    win_size = 0.25
    while True:
        windows_flow, windows_time, windows_rtt = divide_into_windows_with_reltime(flows, times, rtts, win_size)
        tstamp_mins, rtt_mins, window_lens, flow_lens = extract_minimums_and_window_len(windows_flow, windows_time, windows_rtt)

    rtt_mins_allowed = []
    for wlen, rtt in zip(window_lens, rtt_mins):
        if wlen >= 5:
            rtt_mins_allowed.append(rtt)
    
    rtt_pre  = max(rtt_mins_allowed)
    rtt_post = min_dist_post / C_OPTICAL_FIBER_KM_PER_MS
    min_rtt_diff = max(rtt_post - rtt_pre, 0)

    return min_rtt_diff

In [ ]:
countries = sorted(gdf_regions_areas_convex_hull['Country'].tolist())
total = (len(countries) - 1) * len(us_prefixes)
rttdevs_windows = {}
count = 0

for prefix in us_prefixes:
    rttdevs_windows[prefix] = {}
    for country in countries:
        if country == "United States of America":
            continue
        rttdevs_windows[prefix][country] = compute_windowmin_based_deviation(
            gdf_regions_areas_convex_hull, (40.343899, -74.660049), prefix_latlon_map[prefix], country,
            [(flow, time, rtt) for flow, time, rtt in rtts_profiling[prefix]])
        count += 1
        if count % 10000 == 0:
            print(f"{count} pairs done out of {total} ({mu.rnd(count*100/total, 2)}%)")

In [ ]:
coverages_windows_prefix = [prefix_coverage_percentage_per_deviation_threshold(rttdevs_windows, threshold) for threshold in [1] + list(range(25, 101, 25))]

In [ ]:
pu.boxplot(coverages_prefix, {"figsize": (11, 9), "xticks": [1] + list(range(25, 101, 25)), "facecolor": "sns-pastel-6",
                        "xlabel": "Minimum Increase from\nPre-Attack Profiled MinRTT\n(over Windows of Samples) to\nPost-Attack Speed-of-Light RTT (ms)",
                        "ylabel": "% Threat Countries\nAgainst Whom HiDe can\nDefend each US Prefix",
                        "plot_path": "plots/threshold_vs_hide_coverage.png"})

In [ ]:
def compute_maxmin_profiling(prefix, samples):
    win_size = 1
    flows = []
    times = []
    rtts = []
    for f, t, r in samples:
        flows.append(f)
        times.append(t)
        rtts.append(r)
    windows_flow, windows_time, windows_rtt = divide_into_windows_with_reltime(flows, times, rtts, win_size)
    tstamp_mins, rtt_mins, window_lens, flow_lens = extract_minimums_and_window_len(windows_flow, windows_time, windows_rtt)
    rtt_candidates = []
    for r, w in zip(rtt_mins, window_lens):
        if w >= 3:
            rtt_candidates.append(r)
    if len(rtt_candidates) < 1:
        return None
    return max(rtt_candidates)

In [ ]:
prefix_maxmin_map = {}
for prefix in us_ciso_prefixes:
    maxmin = compute_maxmin_profiling(prefix, rtts_profiling[prefix])
    if maxmin:
        prefix_maxmin_map[prefix] = maxmin
    # else:
        # print(prefix)
    # break
print(len(prefix_maxmin_map))

In [ ]:
print(passing_theory.keys())

In [ ]:
passing_defend = {}

In [ ]:
prefixes = sorted(prefix_maxmin_map.keys())
lambdas = [5, 25, 50, 75]

coverages = {}
for l in lambdas:
    passing_defend[l] = {}
    coverages[l] = []
    for prefix in prefixes:
        if prefix in passing_theory:
            post_prefix = []
            maxmin = prefix_maxmin_map[prefix]
            for country in prefix_postrtt_map[prefix]:
                if country in passing_theory[prefix]:
                    post = prefix_postrtt_map[prefix][country]
                    if post - maxmin >= l:
                        if prefix not in passing_defend[l]:
                            passing_defend[l][prefix] = []
                        passing_defend[l][prefix].append(country)
                    post_prefix.append(post)
            count = sum(1 for post in post_prefix if post - maxmin >= l)
            coverages[l].append(count * 100.0 / len(post_prefix))

xs, ys = [], []
for l in lambdas:
    x, y = [], []
    for t in range(101):
        count = sum(1 for coverage in coverages[l] if coverage >= t)
        if len(coverages[l]) > 0:
            y.append(count * 100.0 / len(coverages[l]))
            x.append(t)
    xs.append(x)
    ys.append(y)

pu.lineplots(ys, xs, {
        "figsize": (8, 6),
        # "colors": [sns_colors[0]],
        # "linestyles": ["-"],
        "curvelabels": [f"$\lambda$ = {l} ms" for l in lambdas],
        "loc": "upper right",
        "xlim": (-5, 105),
        "ylim": (-5, 105),
        "loc": "lower left",
        "xlabel": "Prefixes (%)",
        "ylabel": "Optimal Attacks (%)",
        "plot_path": "plots/prefixes_vs_attacks_defendability.pdf"
})

In [ ]:
for i in range(len(lambdas)):
    print(lambdas[i], "ms")
    for p, a in zip(ys[i], xs[i]):
        print(p , a)

In [ ]:
print(passing_defend)

### False positives

In [ ]:
def find_prefixes_with_abs_and_surge(rtts_dict, win_size_time, abs_thresh, surge_thresh):

    attacked_prefixes = {}
    attack_points = {}

    for country in countries:
        attacked_prefixes[country] = []
        attack_points[country] = {}
        
        for prefix in rtts_dict[country]:
            fid_series = [i[0] for i in rtts_dict[country][prefix]]
            ts_series  = [i[1] for i in rtts_dict[country][prefix]]
            tsr_series = [(i - math.floor(ts_series[0])) for i in ts_series]
            rtt_series = [i[2] for i in rtts_dict[country][prefix]]
            flows_1, time_1, rtt_1 = divide_into_windows_with_reltime(fid_series, ts_series, rtt_series, win_size_time[prefix])
            min_time, min_rtts, win_len, flows_len = extract_minimums_and_window_len(flows_1, time_1, rtt_1)
            
            for idx, (t1, t2, r1, r2, w1, w2) in enumerate(zip(
                    min_time[:-1], min_time[1:], min_rtts[:-1], min_rtts[1:], win_len[:-1], win_len[1:])):
                if t2 - t1 <= 2 * win_size_time[prefix] and w1 >= 5 and w2 >= 5:
                    if r1 < abs_thresh[country][prefix] and r2 > abs_thresh[country][prefix] and r2 - r1 > surge_thresh[country][prefix]:
                        attacked_prefixes[country].append(prefix)
                        attack_points[country][prefix] = [(idx, t1, r1), (idx+1, t2, r2)]
                        break

    return attacked_prefixes, attack_points

In [ ]:
rtts_detection_all = {}

for l in lambdas:
    for prefix in passing_defend[l]:
        rtts_detection_all[prefix] = {}
        for country in passing_defend[l][prefix]:
            found = False
            for prefix_j in rtts_detection:
                if prefix == prefix_j:
                    found = True
                    rtts_detection_all[prefix][country] = rtts_detection[prefix]
                    break
            if found:
                break

In [ ]:
for prefix in rtts_detection_all:
    for country in rtts_detection_all[prefix]:
        print(rtts_detection_all[prefix][country])
        break
    break

In [ ]:
attacked_prefixes, attack_points = find_prefixes_with_abs_and_surge(
    rtts_detection_all, selected_countries, window_sizes, absolute_thresholds, surge_thresholds)

In [ ]:
clients_defended = {}
servers_defended = {}
clients_attacked = {}
servers_attacked = {}

for country in prefixes_passing_theory_and_defendability:
    clients_defended[country] = []
    servers_defended[country] = []
    for prefix in prefixes_passing_theory_and_defendability[country]:
        if prefix in prefix_minrtt_puclients:
            clients_defended[country].append(prefix)
        elif prefix in prefix_minrtt_puservers:
            servers_defended[country].append(prefix)

for country in attacked_prefixes:
    clients_attacked[country] = []
    servers_attacked[country] = []
    for prefix in attacked_prefixes[country]:
        if prefix in prefix_minrtt_puclients:
            clients_attacked[country].append(prefix)
        elif prefix in prefix_minrtt_puservers:
            servers_attacked[country].append(prefix)

In [ ]:
for country in selected_countries:
    total_defended = len(clients_defended[country]) + len(servers_defended[country])
    total_attacked = len(clients_attacked[country]) + len(servers_attacked[country])
    print(country, len(clients_attacked[country])/total_attacked, len(servers_attacked[country])/total_attacked)

In [ ]:
pu.barplot(dist_order,
          [len(attacked_prefixes[country])*100/len(prefixes_passing_theory_and_defendability[country]) \
                                                   for country in [d.replace("\n", " ") for d in dist_order]],
          title="", xlabel="Example threat nations", ylabel="False positive rate (%)",
          plot_path = "plots/false_positives.pdf")